# Lab 05 — EXPLAIN e índices (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: ler um plano de execução com `EXPLAIN` e ver o efeito de um índice.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb

con = duckdb.connect()
con.execute('''CREATE TABLE pedidos AS
  SELECT * FROM (VALUES
   (1,'SP','eletronicos',1200.0,1),(2,'SP','livros',50.0,2),(3,'RJ','livros',30.0,1),
   (4,'MG','casa',80.0,3),(5,'SP','eletronicos',800.0,2),(6,'RJ','casa',150.0,4),
   (7,'SP','livros',45.0,1),(8,'MG','eletronicos',600.0,3),(9,'RJ','eletronicos',900.0,2),
   (10,'SP','casa',200.0,5),(11,'MG','livros',25.0,4),(12,'SP','eletronicos',1500.0,1),
   (13,'RJ','livros',60.0,5),(14,'MG','casa',120.0,3),(15,'SP','livros',40.0,2)
  ) AS t(id,estado,categoria,valor,cliente_id)''')
con.execute('SELECT COUNT(*) FROM pedidos').fetchone()

## 1. O plano de execução (EXPLAIN)

In [ ]:
plano = con.execute("EXPLAIN SELECT * FROM pedidos WHERE cliente_id = 1").fetchall()
print('\n'.join(row[1] for row in plano))

## 2. Criar um índice na coluna do filtro

In [ ]:
con.execute('CREATE INDEX idx_cliente ON pedidos(cliente_id)')
print('índice criado — em tabelas grandes, o filtro por cliente_id fica muito mais rápido.')

## 3. Sua vez (mini-desafio)
Traga os **clientes com mais de 3 pedidos** (colunas `cliente_id`, `n`), usando `GROUP BY` + `HAVING`, ordenado por `n` desc. Verifique.

In [ ]:
resposta = con.execute('''
    SELECT cliente_id, COUNT(*) AS n
    FROM pedidos
    GROUP BY cliente_id
    HAVING COUNT(*) > 3
    ORDER BY n DESC, cliente_id
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    try:
        assert rows == [(1, 4), (2, 4)], 'Clientes 1 e 2 têm 4 pedidos cada.'
        print('\u2705 Correto! HAVING filtra grupos depois do GROUP BY.')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)